# Full run of `pd-mice-16s-microbiome` in Google Colab

QIIME 2 (import → DADA2 → taxonomy → SEPP tree → diversity) followed by the R/phyloseq analysis,
on the public data of the QIIME 2 *Parkinson's Mouse* tutorial (48 samples).

**How to use it:** `Runtime → Run all`, and upload `pd-mice-16s-microbiome.zip` when asked.
About **1–1.5 h** (installing QIIME 2 and SEPP are the long parts). Keep the tab open.
At the end, cell 6 prints a summary: copy it and send it to Claude.

In [ ]:
# 1. Upload and unzip the repository
import os, zipfile, glob
from google.colab import files

os.chdir('/content')
uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith('.zip'))
zipfile.ZipFile(zip_name).extractall('/content/repo_zip')
hit = glob.glob('/content/repo_zip/**/scripts/07_phyloseq_analysis.R', recursive=True)
assert hit, 'scripts/07_phyloseq_analysis.R not found in the zip'
REPO = os.path.dirname(os.path.dirname(hit[0]))
os.chdir(REPO)
os.environ['REPO'] = REPO
os.environ['MAMBA_ROOT_PREFIX'] = '/content/micromamba'
print('Repository at:', REPO, '| CPUs:', os.cpu_count())

In [ ]:
%%bash
# 2. Install micromamba, QIIME 2 amplicon 2024.10 and the R environment (~20 min)
set -euo pipefail
cd "$REPO"
command -v micromamba >/dev/null || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
wget -q -O /content/qiime2-amplicon-2024.10.yml https://data.qiime2.org/distro/amplicon/qiime2-amplicon-2024.10-py310-linux-conda.yml
echo '== QIIME 2';  micromamba create -y -n qiime2 -f /content/qiime2-amplicon-2024.10.yml 2>&1 | tail -3
echo '== R';        micromamba create -y -n pd-mice-r -f envs/r-analysis.yml 2>&1 | tail -3
micromamba run -n qiime2 qiime info 2>/dev/null | head -8 || true
micromamba run -n pd-mice-r Rscript -e 'suppressMessages({library(phyloseq);library(vegan);library(lmerTest)}); cat("R packages OK\n")'

In [ ]:
%%bash
# 3. QIIME 2 steps 0-6 + rarefaction plot (~40-60 min; SEPP is the slowest step)
set -euo pipefail
cd "$REPO"
unset MPLBACKEND
micromamba run -n qiime2 bash scripts/run_all.sh qiime

In [ ]:
%%bash
# 4. R / phyloseq analysis
set -euo pipefail
cd "$REPO"
micromamba run -n pd-mice-r bash scripts/run_all.sh r

In [ ]:
# 5. Figures
from IPython.display import Image, display
for f in sorted(glob.glob(f'{REPO}/results/07_phyloseq/figures/*.png')):
    print(os.path.basename(f)); display(Image(f, width=750))

In [ ]:
# 6. SUMMARY — copy everything below and send it to Claude
import pandas as pd
R = f'{REPO}/results'
st = pd.read_csv(f'{R}/02_dada2/stats_export/stats.tsv', sep='\t', comment='#')
st = st[st.iloc[:, 0] != '#q2:types']
for c in st.columns[1:]:
    st[c] = pd.to_numeric(st[c], errors='coerce')
print('### DADA2 per-sample stats (lowest 5 by non-chimeric reads)')
print(st.sort_values(st.columns[-2]).head(5).to_string(index=False))
print('median input / non-chimeric reads:', st['input'].median(), '/', st['non-chimeric'].median())

ft = pd.read_csv(f'{R}/06_export/feature_table.tsv', sep='\t', skiprows=1, index_col=0)
tx = pd.read_csv(f'{R}/06_export/taxonomy.tsv', sep='\t', index_col=0)
print('\n### ASVs:', ft.shape[0], '| samples:', ft.shape[1], '| total reads:', int(ft.values.sum()))
fam = tx.Taxon.str.split(';').apply(lambda x: ';'.join(t.strip() for t in x[:5]))
rel = ft.div(ft.sum(axis=0), axis=1)
top = rel.groupby(fam.reindex(rel.index)).sum().mean(axis=1).sort_values(ascending=False).head(6) * 100
print('\n### Top families (level 5, mean relative abundance %)'); print(top.round(1).to_string())

for f in ['shannon_donor_tests.tsv', 'permanova_last_timepoint.txt']:
    print(f'\n### {f}'); print(open(f'{R}/07_phyloseq/{f}').read())

In [ ]:
%%bash
# 6b. R log (Clostridiales, means, lowest Shannon, Betaproteobacteria) and rarefaction by cage
cd "$REPO"
micromamba run -n pd-mice-r Rscript scripts/07_phyloseq_analysis.R 2>&1 | grep -E 'phyloseq object|Reads per sample|Rarefied|Clostridiales:|Healthy|PD |Lowest|Betaproteobacteria' 
micromamba run -n qiime2 python scripts/08_plot_rarefaction.py results/05_diversity/alpha_rarefaction_export/shannon.csv \
  --group cage_id --out results/07_phyloseq/figures/rarefaction_shannon_cage.png

In [ ]:
# 7. Download figures and tables
!cd "$REPO" && zip -qr /content/pd_mice_results.zip results/07_phyloseq results/02_dada2/stats_export
files.download('/content/pd_mice_results.zip')